In [1]:
from paper.ModelOpt.opt_submodel import Objs
from GaussECG.two_stage_density_peak_clustering import TwoStageDPC
import numpy as np
from Toolbox.DBtool import mitArr
import neurokit2 as nk

item = mitArr().record('100')
signal = np.array(nk.ecg_clean(item.signal, sampling_rate=item.fs, method="vg"))
detector = TwoStageDPC(fs=item.fs)
detector.clc_params(signal)
cluster_param = Objs(rho=detector.params.rho,
                     delta=detector.params.delta,
                     peaks=item.r_loc)

Calculation parameters (in block): 100%|██████████| 120/120 [00:03<00:00, 30.81it/s]


In [ ]:
from paper.ModelOpt.opt_utils import summary, obj_fun
from GaussECG.peak_over_threshold import alarm
import itertools

q1s = np.linspace(0.01, 0.05, 5)
q2s = np.linspace(0.01, 0.05, 5)
params = itertools.product(q1s, q2s)
F1, P, R, M, S = 0, 0, 0, 0, 0
opt_param = None
tol = 5
fs = 360

for param in params:
    delta_params: dict[str, int | float] = {'q': param[0], 'd': int(fs * 0.1), 'quantile': 0.9}
    rho_params: dict[str, int | float] = {'q': param[1], 'd': int(fs * 0.1), 'quantile': 0.9}

    CM = np.zeros((2, 2))
    DIF = []
    rho_peaks = alarm(cluster_param.rho, method='MOM', **rho_params)
    delta_peaks = alarm(cluster_param.delta, method='MOM', **delta_params)
    peaks = np.intersect1d(rho_peaks, delta_peaks)
    cm, dif = obj_fun(cluster_param.peaks, peaks, tol, )
    CM += cm
    DIF = np.append(DIF, dif)
    f1, p, r, m, s = summary(CM, DIF, fs)
    if f1 > F1:
        opt_param = param
    F1, P, R, M, S = f1, p, r, m, s